In [ ]:
import numpy as npimport pandas as pdimport statsmodels.api as smfrom linearmodels.panel import PanelOLSimport matplotlib.pyplot as pltfile_path = "/Users/kylejonespatricia/time_series/north_dakota_production.csv"data = pd.read_csv(file_path)data["ReportDate"] = pd.to_datetime(data["ReportDate"])data = data.set_index(["API_WELLNO", "ReportDate"])data["Days"] = pd.to_numeric(data["Days"], errors="coerce")data = data.dropna(subset=["Days", "Oil"])X_matrix = sm.add_constant(data["Days"])panel_model = PanelOLS(data["Oil"], X_matrix, entity_effects=True).fit(    cov_type="kernel", kernel="bartlett", bandwidth=3)panel_model_clustered = PanelOLS(data["Oil"], X_matrix, entity_effects=True).fit(    cov_type="clustered", cluster_entity=True)dk_se = panel_model.std_errors.to_numpy()cluster_se = panel_model_clustered.std_errors.to_numpy()labels = ["Intercept", "Days"]fig, axes = plt.subplots(1, 2, figsize=(12, 5))sample_wells = data.index.get_level_values("API_WELLNO").unique()[:5]for well in sample_wells:    well_data = data.xs(well, level="API_WELLNO")    axes[0].plot(well_data.index, well_data["Oil"], linewidth=1, alpha=0.7)axes[0].set_xlabel('Date')axes[0].set_ylabel('Oil Production')axes[0].spines['top'].set_visible(False)axes[0].spines['right'].set_visible(False)axes[1].bar([0, 1], dk_se, color='gray', alpha=0.7, label='Driscoll-Kraay SEs', width=0.35)axes[1].bar([0.4, 1.4], cluster_se, color='blue', alpha=0.5, label='Clustered SEs', width=0.35)axes[1].set_xticks([0.2, 1.2])axes[1].set_xticklabels(labels)axes[1].set_ylabel('Standard Error')axes[1].legend(frameon=False)axes[1].spines['top'].set_visible(False)axes[1].spines['right'].set_visible(False)plt.tight_layout()plt.show()print(f"Driscoll-Kraay SEs: {dk_se}")print(f"Clustered SEs: {cluster_se}")plt.title("Monthly Oil Production Distribution")plt.grid(False)plt.savefig("oil_production_boxplot.png")plt.show()import matplotlib.pyplot as pltimport numpy as npfrom PIL import Image, ImageDrawimport os# Define the styleplt.rcParams.update({    "font.family": "serif",    "axes.spines.top": False,    "axes.spines.right": False})# Select 5 sample wellssample_wells = data.index.get_level_values("API_WELLNO").unique()[:5]# Get the time index and rangeall_dates = sorted(data.index.get_level_values("ReportDate").unique())# Directory to save framesframe_dir = "frames"os.makedirs(frame_dir, exist_ok=True)# Create and save frames for the animationframes = []for i, date in enumerate(all_dates):    plt.figure(figsize=(12, 6))        for well in sample_wells:        well_data = data.xs(well, level="API_WELLNO")        well_data = well_data.loc[well_data.index <= date]  # Show data up to the current date        plt.plot(well_data.index, well_data["Oil"], label=f"Well {well}", linewidth=1)        plt.title("Monthly Oil Production Over Time")        plt.grid(False)        # Save frame    frame_path = f"{frame_dir}/frame_{i:03d}.png"    plt.savefig(frame_path)    plt.close()    frames.append(Image.open(frame_path))# Save as GIFgif_path = "oil_production_animation.gif"frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=100, loop=0)print(f"GIF saved at: {gif_path}")data["WellName"].nunique()